
## Folder 05 — File 06: Data Quality Dashboard:


It will provide:

- Overall quality status by dataset.
- Completeness and uniqueness metrics.
- Data type and format failures.
- Business rule failures.
- Quarantine counts.
- A consolidated dashboard-ready Delta table.

**Important**: This creates the data behind a dashboard. It does not automatically create a Databricks SQL dashboard UI.

In [0]:
# 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql import types as T
import uuid

print("Libraries imported successfully.")

# 2. Project configuration

CATALOG = "workspace"
SCHEMA = "ecommerce_silver"

SOURCE_REPORTS = {
    "data_quality": f"{CATALOG}.{SCHEMA}.data_quality_report",
    "completeness_uniqueness": (
        f"{CATALOG}.{SCHEMA}.completeness_uniqueness_report"
    ),
    "data_type_format": (
        f"{CATALOG}.{SCHEMA}.data_type_format_report"
    ),
    "business_rules": (
        f"{CATALOG}.{SCHEMA}.business_rule_validation_report"
    )
}

QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA}.quarantine_records"

DASHBOARD_DETAIL_TABLE = (
    f"{CATALOG}.{SCHEMA}.data_quality_dashboard_detail"
)

DASHBOARD_SUMMARY_TABLE = (
    f"{CATALOG}.{SCHEMA}.data_quality_dashboard_summary"
)

DASHBOARD_OVERALL_TABLE = (
    f"{CATALOG}.{SCHEMA}.data_quality_dashboard_overall"
)

DASHBOARD_FAILURE_TABLE = (
    f"{CATALOG}.{SCHEMA}.data_quality_dashboard_failure_breakdown"
)

DASHBOARD_TOP_ISSUES_TABLE = (
    f"{CATALOG}.{SCHEMA}.data_quality_dashboard_top_issues"
)

DASHBOARD_FRESHNESS_TABLE = (
    f"{CATALOG}.{SCHEMA}.data_quality_dashboard_freshness"
)

DASHBOARD_RUN_ID = str(uuid.uuid4())

print("=" * 75)
print("PROJECT CONFIGURATION")
print("=" * 75)
print(f"Catalog          : {CATALOG}")
print(f"Schema           : {SCHEMA}")
print(f"Dashboard Run ID : {DASHBOARD_RUN_ID}")
print("=" * 75)


# 3. Helper functions

def canonical_dataset_name(dataset_column):

    normalized = F.lower(
        F.trim(dataset_column.cast("string"))
    )

    normalized = F.regexp_replace(
        normalized,
        r"^(silver_|quarantine_)",
        ""
    )

    return (
        F.when(
            normalized.isin(
                "inventory_snapshots",
                "inventory"
            ),
            F.lit("inventory")
        )
        .otherwise(normalized)
    )


def normalize_status_value(status_column):

    status = F.upper(
        F.trim(status_column.cast("string"))
    )

    return (
        F.when(
            status.isin("PASS", "PASSED", "SUCCESS"),
            F.lit("PASS")
        )
        .when(
            status.isin("FAIL", "FAILED", "ERROR"),
            F.lit("FAIL")
        )
        .when(
            status.isin("WARN", "WARNING"),
            F.lit("WARN")
        )
        .when(
            status.isin("INFO", "INFORMATION", "INFORMATIONAL"),
            F.lit("INFO")
        )
        .when(
            status.isin("REVIEW", "UNKNOWN", "PENDING"),
            F.lit("REVIEW")
        )
        .otherwise(F.lit("REVIEW"))
    )


def get_report_timestamp_column(df):

    candidates = [
        "checked_at",
        "validation_timestamp",
        "summary_generated_at",
        "created_at",
        "generated_at"
    ]

    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    return None


def first_available_column(df, candidates, default_value=None):

    available_columns = set(df.columns)

    for candidate in candidates:
        if candidate in available_columns:
            return F.col(candidate)

    return F.lit(default_value)


# 4. Select the exact latest report batch

def select_latest_report_batch(df, source_name):

    timestamp_column = get_report_timestamp_column(df)

    if timestamp_column is None:

        print(
            f"WARNING: {source_name} has no timestamp column."
        )

        return None, None, 0, "MISSING_TIMESTAMP_COLUMN"

    timestamp_df = df.withColumn(
        "_source_timestamp",
        F.col(timestamp_column).cast("timestamp")
    )

    stats = timestamp_df.agg(
        F.count("*").alias("total_rows"),
        F.count("_source_timestamp").alias("valid_timestamps"),
        F.max("_source_timestamp").alias("latest_timestamp"),
        F.min("_source_timestamp").alias("earliest_timestamp")
    ).first()

    total_rows = stats["total_rows"]
    valid_timestamps = stats["valid_timestamps"]
    latest_timestamp = stats["latest_timestamp"]
    earliest_timestamp = stats["earliest_timestamp"]

    print("-" * 75)
    print(f"Source report      : {source_name}")
    print(f"Historical rows    : {total_rows}")
    print(f"Valid timestamps   : {valid_timestamps}")
    print(f"Earliest timestamp : {earliest_timestamp}")
    print(f"Latest timestamp   : {latest_timestamp}")
    print("-" * 75)

    if total_rows == 0:
        return None, None, 0, "EMPTY_REPORT"

    if valid_timestamps == 0:
        return None, None, 0, "INVALID_TIMESTAMP"

    if valid_timestamps != total_rows:
        return None, latest_timestamp, 0, "PARTIAL_INVALID_TIMESTAMP"

    # Select only rows with the exact latest timestamp.

    # This prevents earlier validation runs within the same

    # 60-minute period from entering the dashboard.

    latest_df = (
        timestamp_df
        .filter(
            F.col("_source_timestamp") == F.lit(latest_timestamp)
        )
        .drop("_source_timestamp")
    )

    selected_count = latest_df.count()

    if selected_count == 0:
        return None, latest_timestamp, 0, "NO_LATEST_BATCH_ROWS"

    print(f"Selected latest rows: {selected_count}")
    print(f"Selected timestamp  : {latest_timestamp}")

    return (
        latest_df,
        latest_timestamp,
        selected_count,
        "LOADED"
    )


# 5. Normalize source report schemas

def normalize_report(df, source_name):

    dataset_column = first_available_column(
        df,
        ["dataset_name", "_quarantine_dataset", "table_name"],
        "unknown"
    )

    # Business-rule report uses rule_name.

    # Other validation reports generally use check_name.

    check_column = first_available_column(
        df,
        ["check_name", "rule_name", "_quarantine_reason"],
        None
    )

    column_column = first_available_column(
        df,
        ["column_name"],
        None
    )

    total_column = first_available_column(
        df,
        ["total_records", "total_count"],
        None
    )

    failed_column = first_available_column(
        df,
        [
            "failed_records",
            "invalid_records",
            "quarantined_records"
        ],
        0
    )

    percentage_column = first_available_column(
        df,
        ["failure_percentage"],
        None
    )

    status_column = first_available_column(
        df,
        ["status", "validation_status", "result_status"],
        None
    )

    details_column = first_available_column(
        df,
        ["details", "description", "rule_description"],
        None
    )

    timestamp_column = first_available_column(
        df,
        [
            "checked_at",
            "validation_timestamp",
            "summary_generated_at",
            "created_at",
            "generated_at"
        ],
        None
    )

    normalized = df.select(
        dataset_column.cast("string").alias("dataset_name"),

        canonical_dataset_name(
            dataset_column
        ).alias("dataset_group"),

        check_column.cast("string").alias("_raw_check_name"),

        column_column.cast("string").alias("column_name"),

        F.lit(source_name).alias("source_report"),

        total_column.cast("long").alias("total_records"),

        F.coalesce(
            failed_column.cast("long"),
            F.lit(0)
        ).alias("failed_records"),

        percentage_column.cast("double").alias(
            "_source_failure_percentage"
        ),

        status_column.cast("string").alias("_source_status"),

        details_column.cast("string").alias("details"),

        timestamp_column.cast("timestamp").alias("checked_at")
    )

    # Normalize check names.

    normalized = normalized.withColumn(
        "check_name",
        F.when(
            F.col("_raw_check_name").isNull()
            | (F.trim(F.col("_raw_check_name")) == ""),
            F.lit("UNNAMED_CHECK")
        ).otherwise(
            F.trim(F.col("_raw_check_name"))
        )
    )

    # Append column name when available.

    normalized = normalized.withColumn(
        "check_name",
        F.when(
            F.col("column_name").isNotNull()
            & (F.trim(F.col("column_name")) != ""),
            F.concat(
                F.col("check_name"),
                F.lit(" - "),
                F.col("column_name")
            )
        ).otherwise(
            F.col("check_name")
        )
    )

    # Calculate failure percentage if source value is missing.

    normalized = normalized.withColumn(
        "failure_percentage",
        F.coalesce(
            F.col("_source_failure_percentage"),
            F.when(
                F.col("total_records") > 0,
                F.round(
                    F.col("failed_records")
                    / F.col("total_records") * 100,
                    2
                )
            ).otherwise(F.lit(0.0))
        )
    )

    # Normalize status.

    normalized = normalized.withColumn(
        "status",
        F.when(
            F.col("_source_status").isNotNull()
            & (F.trim(F.col("_source_status")) != ""),
            normalize_status_value(
                F.col("_source_status")
            )
        ).otherwise(
            F.when(
                F.col("failed_records") > 0,
                F.lit("REVIEW")
            ).otherwise(F.lit("PASS"))
        )
    )

    normalized = normalized.withColumn(
        "status_basis",
        F.when(
            F.col("_source_status").isNotNull()
            & (F.trim(F.col("_source_status")) != ""),
            F.lit("SOURCE_REPORT")
        ).otherwise(
            F.lit("INFERRED_FROM_FAILURE_COUNT")
        )
    )

    return normalized.select(
        "dataset_name",
        "dataset_group",
        "check_name",
        "column_name",
        "source_report",
        "total_records",
        "failed_records",
        "failure_percentage",
        "details",
        "checked_at",
        "status",
        "status_basis"
    )


# 6. Load all source reports

normalized_reports = []
source_metadata = []

for source_name, table_name in SOURCE_REPORTS.items():

    print("=" * 75)
    print(f"PROCESSING: {source_name}")
    print(f"TABLE: {table_name}")

    if not spark.catalog.tableExists(table_name):

        print(f"WARNING: Missing table: {table_name}")

        source_metadata.append((
            source_name,
            table_name,
            None,
            0,
            "TABLE_NOT_FOUND"
        ))

        continue

    source_df = spark.table(table_name)

    latest_df, latest_timestamp, selected_count, load_status = (
        select_latest_report_batch(
            source_df,
            source_name
        )
    )

    source_metadata.append((
        source_name,
        table_name,
        latest_timestamp,
        int(selected_count),
        load_status
    ))

    if load_status != "LOADED":

        print(
            f"WARNING: {source_name} skipped. "
            f"Reason: {load_status}"
        )

        continue

    normalized_df = normalize_report(
        latest_df,
        source_name
    )

    normalized_reports.append(normalized_df)

    print(f"Successfully loaded: {table_name}")


if not normalized_reports:

    raise Exception(
        "No valid validation reports could be loaded. "
        "Run the upstream validation notebooks."
    )

print("=" * 75)
print(f"Valid reports loaded: {len(normalized_reports)}")
print("=" * 75)


# 7. Combine reports

dashboard_detail_df = normalized_reports[0]

for report_df in normalized_reports[1:]:

    dashboard_detail_df = dashboard_detail_df.unionByName(
        report_df
    )

dashboard_detail_df = (
    dashboard_detail_df
    .withColumn(
        "dashboard_run_id",
        F.lit(DASHBOARD_RUN_ID)
    )
    .withColumn(
        "dashboard_generated_at",
        F.current_timestamp()
    )
)

print("Combined dashboard detail:")

display(
    dashboard_detail_df.orderBy(
        "dataset_group",
        "source_report",
        "check_name"
    )
)


# 8. Check unnamed validation rules

unnamed_checks_df = (
    dashboard_detail_df
    .filter(
        F.col("check_name") == "UNNAMED_CHECK"
    )
)

unnamed_count = unnamed_checks_df.count()

print(f"Unnamed check count: {unnamed_count}")

if unnamed_count > 0:

    print(
        "WARNING: Some source reports contain blank check names."
    )

    display(unnamed_checks_df)


# 9. Source report freshness

freshness_schema = T.StructType([
    T.StructField("source_report", T.StringType(), True),
    T.StructField("table_name", T.StringType(), True),
    T.StructField("latest_checked_at", T.TimestampType(), True),
    T.StructField("selected_rows", T.LongType(), True),
    T.StructField("load_status", T.StringType(), True)
])

source_freshness_df = spark.createDataFrame(
    source_metadata,
    freshness_schema
)

source_freshness_df = (
    source_freshness_df
    .withColumn(
        "dashboard_run_id",
        F.lit(DASHBOARD_RUN_ID)
    )
    .withColumn(
        "dashboard_generated_at",
        F.current_timestamp()
    )
    .withColumn(
        "source_age_minutes",
        F.when(
            F.col("latest_checked_at").isNotNull(),
            (
                F.unix_timestamp(F.current_timestamp())
                - F.unix_timestamp(F.col("latest_checked_at"))
            ) / 60
        )
    )
    .withColumn(
        "freshness_status",
        F.when(
            F.col("load_status") != "LOADED",
            F.col("load_status")
        )
        .when(
            F.col("source_age_minutes") < 0,
            F.lit("FUTURE_TIMESTAMP")
        )
        .when(
            F.col("source_age_minutes") > 180,
            F.lit("STALE")
        )
        .otherwise(F.lit("RECENT"))
    )
)

print("SOURCE FRESHNESS:")
display(source_freshness_df)


# 10. Current quarantine counts

if spark.catalog.tableExists(QUARANTINE_TABLE):

    quarantine_source_df = spark.table(
        QUARANTINE_TABLE
    )

    if "_quarantine_dataset" not in quarantine_source_df.columns:

        raise Exception(
            "quarantine_records is missing _quarantine_dataset."
        )

    quarantine_counts_df = (
        quarantine_source_df
        .groupBy("_quarantine_dataset")
        .agg(
            F.count("*").cast("long").alias(
                "current_quarantined_records"
            )
        )
        .withColumn(
            "dataset_group",
            canonical_dataset_name(
                F.col("_quarantine_dataset")
            )
        )
        .groupBy("dataset_group")
        .agg(
            F.sum("current_quarantined_records").cast("long").alias(
                "current_quarantined_records"
            )
        )
    )

else:

    print("WARNING: Quarantine table not found.")

    quarantine_counts_df = spark.createDataFrame(
        [],
        "dataset_group STRING, current_quarantined_records LONG"
    )


# 11. Dataset summary

dataset_summary_df = (
    dashboard_detail_df
    .groupBy("dataset_group")
    .agg(
        F.count("*").cast("long").alias("total_checks"),

        F.sum(
            F.when(F.col("status") == "PASS", 1).otherwise(0)
        ).cast("long").alias("passed_checks"),

        F.sum(
            F.when(F.col("status") == "FAIL", 1).otherwise(0)
        ).cast("long").alias("failed_checks"),

        F.sum(
            F.when(F.col("status") == "WARN", 1).otherwise(0)
        ).cast("long").alias("warning_checks"),

        F.sum(
            F.when(F.col("status") == "REVIEW", 1).otherwise(0)
        ).cast("long").alias("review_checks"),

        F.sum(
            F.when(F.col("status") == "INFO", 1).otherwise(0)
        ).cast("long").alias("informational_checks"),

        F.sum("failed_records").cast("long").alias(
            "sum_of_reported_failed_records_non_distinct"
        ),

        F.max("checked_at").alias("last_checked_at")
    )
    .withColumn(
        "applicable_checks",
        F.col("total_checks") - F.col("informational_checks")
    )
    .withColumn(
        "check_pass_rate",
        F.when(
            F.col("applicable_checks") > 0,
            F.round(
                F.col("passed_checks")
                / F.col("applicable_checks") * 100,
                2
            )
        ).otherwise(F.lit(100.0))
    )
    .join(
        quarantine_counts_df,
        on="dataset_group",
        how="left"
    )
    .withColumn(
        "current_quarantined_records",
        F.coalesce(
            F.col("current_quarantined_records"),
            F.lit(0)
        )
    )
    .withColumn(
        "overall_status",
        F.when(F.col("failed_checks") > 0, "FAIL")
        .when(F.col("review_checks") > 0, "REVIEW")
        .when(F.col("warning_checks") > 0, "WARN")
        .otherwise("PASS")
    )
    .withColumn(
        "dashboard_run_id",
        F.lit(DASHBOARD_RUN_ID)
    )
    .withColumn(
        "dashboard_generated_at",
        F.current_timestamp()
    )
)

print("DATASET SUMMARY:")
display(dataset_summary_df.orderBy("dataset_group"))


# 12. Overall dashboard KPIs

overall_summary_df = (
    dashboard_detail_df
    .agg(
        F.count("*").cast("long").alias("total_checks"),

        F.sum(
            F.when(F.col("status") == "PASS", 1).otherwise(0)
        ).cast("long").alias("passed_checks"),

        F.sum(
            F.when(F.col("status") == "FAIL", 1).otherwise(0)
        ).cast("long").alias("failed_checks"),

        F.sum(
            F.when(F.col("status") == "WARN", 1).otherwise(0)
        ).cast("long").alias("warning_checks"),

        F.sum(
            F.when(F.col("status") == "REVIEW", 1).otherwise(0)
        ).cast("long").alias("review_checks"),

        F.sum(
            F.when(F.col("status") == "INFO", 1).otherwise(0)
        ).cast("long").alias("informational_checks"),

        F.sum("failed_records").cast("long").alias(
            "sum_of_reported_failed_records_non_distinct"
        ),

        F.max("checked_at").alias(
            "latest_source_check_timestamp"
        )
    )
    .withColumn(
        "applicable_checks",
        F.col("total_checks") - F.col("informational_checks")
    )
    .withColumn(
        "check_pass_rate",
        F.when(
            F.col("applicable_checks") > 0,
            F.round(
                F.col("passed_checks")
                / F.col("applicable_checks") * 100,
                2
            )
        ).otherwise(F.lit(100.0))
    )
    .withColumn(
        "overall_status",
        F.when(F.col("failed_checks") > 0, "FAIL")
        .when(F.col("review_checks") > 0, "REVIEW")
        .when(F.col("warning_checks") > 0, "WARN")
        .otherwise("PASS")
    )
    .withColumn(
        "dashboard_run_id",
        F.lit(DASHBOARD_RUN_ID)
    )
    .withColumn(
        "dashboard_generated_at",
        F.current_timestamp()
    )
)

print("OVERALL DASHBOARD KPIs:")
display(overall_summary_df)


# 13. Failure breakdown

failure_breakdown_df = (
    dashboard_detail_df
    .groupBy("source_report", "status")
    .agg(
        F.count("*").cast("long").alias("number_of_checks"),

        F.sum("failed_records").cast("long").alias(
            "sum_of_reported_failed_records_non_distinct"
        )
    )
    .withColumn(
        "dashboard_run_id",
        F.lit(DASHBOARD_RUN_ID)
    )
    .withColumn(
        "dashboard_generated_at",
        F.current_timestamp()
    )
    .orderBy("source_report", "status")
)

display(failure_breakdown_df)


# 14. Top issues

top_issues_df = (
    dashboard_detail_df
    .filter(
        F.col("status").isin(
            "FAIL",
            "WARN",
            "REVIEW"
        )
    )
    .groupBy(
        "dataset_group",
        "dataset_name",
        "check_name",
        "source_report",
        "status",
        "status_basis",
        "details"
    )
    .agg(
        F.sum("failed_records").cast("long").alias(
            "sum_of_reported_failed_records_non_distinct"
        ),

        F.count("*").cast("long").alias("check_occurrences"),

        F.max("checked_at").alias("last_checked_at")
    )
    .withColumn(
        "dashboard_run_id",
        F.lit(DASHBOARD_RUN_ID)
    )
    .withColumn(
        "dashboard_generated_at",
        F.current_timestamp()
    )
    .orderBy(
        F.when(F.col("status") == "FAIL", 1)
        .when(F.col("status") == "REVIEW", 2)
        .otherwise(3),
        F.desc("sum_of_reported_failed_records_non_distinct")
    )
)

display(top_issues_df)


# 15. Dashboard validation

required_columns = [
    "dataset_name",
    "dataset_group",
    "check_name",
    "source_report",
    "total_records",
    "failed_records",
    "failure_percentage",
    "details",
    "checked_at",
    "status",
    "status_basis",
    "dashboard_run_id",
    "dashboard_generated_at"
]

missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in dashboard_detail_df.columns
]

if missing_columns:

    raise Exception(
        f"Dashboard detail missing columns: {missing_columns}"
    )

invalid_status_count = (
    dashboard_detail_df
    .filter(
        ~F.col("status").isin(
            "PASS",
            "FAIL",
            "WARN",
            "INFO",
            "REVIEW"
        )
    )
    .count()
)

if invalid_status_count > 0:

    raise Exception(
        f"Invalid statuses found: {invalid_status_count}"
    )

null_dataset_count = (
    dashboard_detail_df
    .filter(
        F.col("dataset_group").isNull()
        | (F.trim(F.col("dataset_group")) == "")
    )
    .count()
)

if null_dataset_count > 0:

    raise Exception(
        f"Invalid dataset groups found: {null_dataset_count}"
    )

if dashboard_detail_df.count() == 0:

    raise Exception("Dashboard detail contains no records.")

print("Dashboard validation completed successfully.")


# 16. Save dashboard detail

(
    dashboard_detail_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DASHBOARD_DETAIL_TABLE)
)

print(f"Saved: {DASHBOARD_DETAIL_TABLE}")


# 17. Save dataset summary

(
    dataset_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DASHBOARD_SUMMARY_TABLE)
)

print(f"Saved: {DASHBOARD_SUMMARY_TABLE}")


# 18. Save overall summary

(
    overall_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DASHBOARD_OVERALL_TABLE)
)

print(f"Saved: {DASHBOARD_OVERALL_TABLE}")


# 19. Save failure breakdown

(
    failure_breakdown_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DASHBOARD_FAILURE_TABLE)
)

print(f"Saved: {DASHBOARD_FAILURE_TABLE}")


# 20. Save top issues

(
    top_issues_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DASHBOARD_TOP_ISSUES_TABLE)
)

print(f"Saved: {DASHBOARD_TOP_ISSUES_TABLE}")


# 21. Save source freshness

(
    source_freshness_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DASHBOARD_FRESHNESS_TABLE)
)

print(f"Saved: {DASHBOARD_FRESHNESS_TABLE}")


# 22. Final output

print("=" * 75)
print("E-COMMERCE DATA QUALITY DASHBOARD")
print("=" * 75)

print(f"Dashboard Run ID: {DASHBOARD_RUN_ID}")

print("\nOVERALL KPIs")
display(spark.table(DASHBOARD_OVERALL_TABLE))

print("\nDATASET SUMMARY")
display(spark.table(DASHBOARD_SUMMARY_TABLE))

print("\nFAILURE BREAKDOWN")
display(spark.table(DASHBOARD_FAILURE_TABLE))

print("\nTOP ISSUES")
display(spark.table(DASHBOARD_TOP_ISSUES_TABLE))

print("\nSOURCE FRESHNESS")
display(spark.table(DASHBOARD_FRESHNESS_TABLE))

print("\nDASHBOARD DETAIL")
display(spark.table(DASHBOARD_DETAIL_TABLE))

print("=" * 75)
print("DASHBOARD PREPARATION COMPLETED")
print("=" * 75)

print(
    "\nIMPORTANT: Source validation reports were not modified."
)

print(
    "Only the exact latest checked_at timestamp was selected "
    "from each source report."
)

print(
    "Dashboard generation time is different from "
    "source validation time."
)

print(
    "Dashboard tables contain only the latest dashboard run "
    "because they are overwritten during each execution."
)


In [0]:
%sql
SELECT
    source_report,
    load_status,
    latest_checked_at,
    selected_rows
FROM workspace.ecommerce_silver.data_quality_dashboard_freshness
ORDER BY source_report;

In [0]:
%sql
SELECT COUNT(*) AS total_rows
FROM workspace.ecommerce_silver.business_rule_validation_report;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_silver.business_rule_validation_report;


### Useful SQL queries for the dashboard:


Run these in Databricks SQL after executing the notebook.

In [0]:
%sql
SELECT
    MAX(checked_at) AS latest_validation_time,
    COUNT(*) AS total_report_rows
FROM workspace.ecommerce_silver.business_rule_validation_report;


**Query 1 — Overall dataset quality:**

In [0]:
%sql
SELECT
    dataset_name,
    rule_name,
    severity,
    status,
    total_records,
    failed_records,
    checked_at
FROM workspace.ecommerce_silver.business_rule_validation_report
ORDER BY checked_at DESC;


### Query 2 — Failed validations:

In [0]:
%sql
SELECT
    dataset_name,
    check_name,
    source_report,
    failed_records,
    failure_percentage,
    details
FROM workspace.ecommerce_silver.data_quality_dashboard_detail
WHERE status = 'FAIL'
ORDER BY failed_records DESC;


### Query 3 — Failures by validation category:

In [0]:
%sql
SELECT
    source_report,
    status,
    COUNT(*) AS total_checks,
    SUM(failed_records) AS failed_records
FROM workspace.ecommerce_silver.data_quality_dashboard_detail
GROUP BY source_report, status
ORDER BY source_report, status;

In [0]:
%sql

DESCRIBE TABLE
workspace.ecommerce_silver.quarantine_dataset_statistics;

In [0]:
%sql

SHOW TABLES IN workspace.ecommerce_silver;

In [0]:
%sql

DESCRIBE TABLE workspace.ecommerce_silver.data_quality_report;

DESCRIBE TABLE workspace.ecommerce_silver.data_quality_dashboard_detail;

DESCRIBE TABLE workspace.ecommerce_silver.data_quality_dashboard_failure_breakdown;

DESCRIBE TABLE workspace.ecommerce_silver.data_type_format_report;

In [0]:
%sql

SELECT *
FROM workspace.ecommerce_silver.data_quality_report
WHERE UPPER(status) = 'FAIL'
ORDER BY dataset_name;

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT dataset_name) AS datasets,
    MAX(checked_at) AS latest_check
FROM workspace.ecommerce_silver.data_quality_report;

In [0]:
%sql

SELECT
    status,
    COUNT(*) AS check_count,
    MAX(checked_at) AS latest_check
FROM workspace.ecommerce_silver.data_quality_report
GROUP BY status
ORDER BY status;

In [0]:
%sql

SELECT *
FROM workspace.ecommerce_silver.data_quality_report
ORDER BY checked_at DESC, dataset_name, check_name
LIMIT 100;

In [0]:
%sql

SELECT *
FROM workspace.ecommerce_silver.data_quality_dashboard_failure_breakdown
LIMIT 100;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_silver.business_rule_validation_report;

In [0]:
%sql
SELECT
    dataset_name,
    rule_name,
    severity,
    status,
    total_records,
    failed_records,
    failure_percentage,
    description,
    checked_at
FROM workspace.ecommerce_silver.business_rule_validation_report
WHERE UPPER(status) = 'FAIL'
ORDER BY checked_at DESC, dataset_name, rule_name;

In [0]:
%sql
SELECT
    status,
    severity,
    COUNT(*) AS total_rules,
    SUM(failed_records) AS total_failed_records,
    MAX(checked_at) AS latest_check
FROM workspace.ecommerce_silver.business_rule_validation_report
GROUP BY status, severity
ORDER BY status, severity;

In [0]:
%sql
SELECT
    checked_at,
    COUNT(*) AS total_rules,
    SUM(CASE WHEN UPPER(status) = 'FAIL' THEN 1 ELSE 0 END) AS failed_rules,
    SUM(failed_records) AS total_failed_records
FROM workspace.ecommerce_silver.business_rule_validation_report
GROUP BY checked_at
ORDER BY checked_at DESC;

In [0]:
%sql

WITH current_checks AS (

    SELECT
        'orders_without_customers' AS rule_name,
        COUNT(*) AS failed_records
    FROM workspace.ecommerce_silver.silver_orders o
    LEFT ANTI JOIN workspace.ecommerce_silver.silver_customers c
        ON o.customer_id = c.customer_id

    UNION ALL

    SELECT
        'order_items_without_orders',
        COUNT(*)
    FROM workspace.ecommerce_silver.silver_order_items i
    LEFT ANTI JOIN workspace.ecommerce_silver.silver_orders o
        ON i.order_id = o.order_id

    UNION ALL

    SELECT
        'order_items_without_products',
        COUNT(*)
    FROM workspace.ecommerce_silver.silver_order_items i
    LEFT ANTI JOIN workspace.ecommerce_silver.silver_products p
        ON i.product_id = p.product_id

    UNION ALL

    SELECT
        'payments_without_orders',
        COUNT(*)
    FROM workspace.ecommerce_silver.silver_payments p
    LEFT ANTI JOIN workspace.ecommerce_silver.silver_orders o
        ON p.order_id = o.order_id

    UNION ALL

    SELECT
        'shipments_without_orders',
        COUNT(*)
    FROM workspace.ecommerce_silver.silver_shipments s
    LEFT ANTI JOIN workspace.ecommerce_silver.silver_orders o
        ON s.order_id = o.order_id

    UNION ALL

    SELECT
        'inventory_without_products',
        COUNT(*)
    FROM workspace.ecommerce_silver.silver_inventory i
    LEFT ANTI JOIN workspace.ecommerce_silver.silver_products p
        ON i.product_id = p.product_id

    UNION ALL

    SELECT
        'orders_without_items',
        COUNT(*)
    FROM workspace.ecommerce_silver.silver_orders o
    WHERE UPPER(o.order_status) <> 'CANCELLED'
      AND NOT EXISTS (
          SELECT 1
          FROM workspace.ecommerce_silver.silver_order_items i
          INNER JOIN workspace.ecommerce_silver.silver_products p
              ON i.product_id = p.product_id
          WHERE i.order_id = o.order_id
      )
)

SELECT
    rule_name,
    failed_records,
    CASE
        WHEN failed_records = 0 THEN 'PASS'
        ELSE 'FAIL'
    END AS status
FROM current_checks
ORDER BY rule_name;

In [0]:
%sql
SELECT
    dataset_name,
    rule_name,
    severity,
    status,
    total_records,
    failed_records,
    description,
    checked_at
FROM workspace.ecommerce_silver.business_rule_validation_report
ORDER BY checked_at DESC, dataset_name, rule_name;

In [0]:
%sql
SELECT
    status,
    COUNT(*) AS total_rules,
    SUM(failed_records) AS failed_records
FROM workspace.ecommerce_silver.business_rule_validation_report
GROUP BY status
ORDER BY status;

In [0]:
%sql
DESCRIBE TABLE
workspace.ecommerce_silver.data_quality_dashboard_summary;

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_silver.data_quality_dashboard_summary;

In [0]:
%sql
DESCRIBE TABLE
workspace.ecommerce_silver.business_rule_validation_report;

In [0]:
%sql
SELECT
    'business_rule_validation_report' AS report_name,
    COUNT(*) AS total_rows,
    MAX(checked_at) AS latest_validation
FROM workspace.ecommerce_silver.business_rule_validation_report

UNION ALL

SELECT
    'data_quality_report',
    COUNT(*),
    MAX(checked_at)
FROM workspace.ecommerce_silver.data_quality_report

UNION ALL

SELECT
    'completeness_uniqueness_report',
    COUNT(*),
    MAX(checked_at)
FROM workspace.ecommerce_silver.completeness_uniqueness_report

UNION ALL

SELECT
    'data_type_format_report',
    COUNT(*),
    MAX(checked_at)
FROM workspace.ecommerce_silver.data_type_format_report;

In [0]:
%sql
DESCRIBE TABLE
workspace.ecommerce_silver.data_quality_dashboard_detail;

In [0]:
%sql

DESCRIBE TABLE
%sqworkspace.ecommerce_silver.data_quality_dashboard_failure_breakdown;

In [0]:
%sql
TRUNCATE TABLE workspace.ecommerce_silver.business_rule_validation_report;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_silver.business_rule_validation_report;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_silver.completeness_uniqueness_report;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_silver.data_quality_report;

In [0]:
%sql

DESCRIBE TABLE workspace.ecommerce_silver.completeness_uniqueness_report;

DESCRIBE TABLE workspace.ecommerce_silver.data_quality_report;

DESCRIBE TABLE workspace.ecommerce_silver.data_type_format_report;

In [0]:
%sql

DESCRIBE TABLE workspace.ecommerce_silver.data_quality_report;

DESCRIBE TABLE workspace.ecommerce_silver.data_type_format_report;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_silver.data_type_format_report;